# Strategy Benchmark

Extracts the top-N best performing strategies from all `model_comparison_results*.csv` files (output of `model_comparison.ipynb` / `model_comparison_mlx.ipynb`, fine-tuned models only), then re-runs each on the FinAgent benchmark stocks and date range.

**Reusable**: re-run this notebook any time those CSVs are updated — it always picks up the latest best strategies.

### Installations

In [ ]:
%pip install plotly pandas -q

# Shared backtest / prompt / reward code: the `trading_rl` package in this repo
import os, sys
if os.path.isdir('../trading_rl'):   # local clone: import straight from the repo
    sys.path.insert(0, os.path.abspath('..'))
else:                                # Colab / Kaggle
    %pip install -q "trading-rl @ git+https://github.com/adhamhelmy/llm-fine-tuning.git"

### Config

In [ ]:
ALPACA_API_KEY    = ''
ALPACA_SECRET_KEY = ''

RESULTS_GLOB      = 'model_comparison_results*.csv'  # all matching files are concatenated

# FinAgent benchmark
BENCHMARK_SYMBOLS = ['AAPL', 'AMZN', 'MSFT', 'TSLA', 'GOOGL']
BENCHMARK_START   = '2022-06-01'
BENCHMARK_END     = '2024-01-01'

# Which fine-tuned models to extract strategies from
TARGET_MODELS = [
    'Qwen2.5-7B LoRA v1-500',
    'Llama-3.1-8B LoRA v1-500',
    'Qwen2.5-32B LoRA v1-500',
]

# How many top strategies to extract per model (ranked by return_pct)
TOP_N = 10

### Imports & Backtester

In [ ]:
import glob

import pandas as pd

from trading_rl import Backtester, extract_strategy

In [ ]:
bt_instance = Backtester(ALPACA_API_KEY, ALPACA_SECRET_KEY, verbose=False)
bt_instance.load_bars(BENCHMARK_SYMBOLS, BENCHMARK_START, BENCHMARK_END)

### Extract Best Strategies from CSV

In [ ]:
results_files = sorted(glob.glob(RESULTS_GLOB))
assert results_files, f'No files match {RESULTS_GLOB} — run model_comparison(_mlx).ipynb first'
df_results = pd.concat([pd.read_csv(p) for p in results_files], ignore_index=True)
print(f'Loaded {len(df_results)} rows from {results_files}')

extracted = []
for model in TARGET_MODELS:
    for symbol in BENCHMARK_SYMBOLS:
        candidates = df_results[
            (df_results['model']         == model)  &
            (df_results['symbol']        == symbol) &
            (df_results['status']        == 'profitable') &
            (df_results['strategy_code'].notna())
        ].sort_values('avg_annual_return_pct', ascending=False)

        if candidates.empty:
            print(f'  SKIP {model} [{symbol}]: no profitable strategies found')
            continue

        for rank, (_, row) in enumerate(candidates.head(TOP_N).iterrows(), 1):
            extracted.append({
                'model':      model,
                'symbol':     symbol,
                'rank':       rank,
                'origin_arr': row['avg_annual_return_pct'],
                'origin_shr': row['sharpe_ratio'],
                'code':       row['strategy_code'],
                'label':      f"{model} [{symbol}] #{rank}",
            })

print(f'\nExtracted {len(extracted)} candidates (top {TOP_N} per model × symbol):\n')
for s in extracted:
    print(f"  {s['label']}  origin ARR={s['origin_arr']:.1f}%")


### Run on Benchmark Symbols

In [ ]:
all_results = []

for strat_info in extracted:
    symbol = strat_info['symbol']
    try:
        strategy_cls = extract_strategy(strat_info['code'])
    except Exception as e:
        print(f"LOAD ERROR {strat_info['label']}: {e}")
        all_results.append({
            'strategy': strat_info['label'], 'model': strat_info['model'],
            'symbol': symbol, 'rank': strat_info['rank'],
            'origin_arr': strat_info['origin_arr'],
            'return_pct': None, 'sharpe_ratio': None,
            'avg_annual': None, 'max_drawdown': None, 'status': 'error',
        })
        continue

    print(f"Running: {strat_info['label']}", end='  ')
    try:
        ret, sharpe, avg_ann, max_dd = bt_instance.run_with_timeout(
            strategy_cls, symbol, BENCHMARK_START, BENCHMARK_END
        )
        status = 'profitable' if ret > 0 else ('no_trades' if ret == 0 and sharpe is None else 'loss')
        print(f"ARR={avg_ann:.1f}%  SHR={sharpe}  MDD={max_dd:.1f}%  [{status}]")
    except Exception as e:
        ret, sharpe, avg_ann, max_dd, status = None, None, None, None, 'error'
        print(f"ERROR — {str(e)[:80]}")

    all_results.append({
        'strategy':     strat_info['label'],
        'model':        strat_info['model'],
        'symbol':       symbol,
        'rank':         strat_info['rank'],
        'origin_arr':   strat_info['origin_arr'],
        'return_pct':   ret,
        'sharpe_ratio': sharpe,
        'avg_annual':   avg_ann,
        'max_drawdown': max_dd,
        'status':       status,
    })

df_all = pd.DataFrame(all_results)

# Keep best per model × symbol: profitable > loss > no_trades/error, then highest avg_annual
status_priority = {'profitable': 0, 'loss': 1, 'no_trades': 2, 'error': 3}
df_all['_sp'] = df_all['status'].map(status_priority).fillna(3)
df_bench = (
    df_all
    .sort_values(['model', 'symbol', '_sp', 'avg_annual'], ascending=[True, True, True, False])
    .groupby(['model', 'symbol'])
    .first()
    .reset_index()
    .drop(columns='_sp')
)

print(f'\nDone. {len(df_all)} candidates → {len(df_bench)} best results.')


### Results Table

In [ ]:
for metric, col in [
    ('ARR — Avg Annual Return %', 'avg_annual'),
    ('SHR — Sharpe Ratio',        'sharpe_ratio'),
    ('MDD — Max Drawdown %',      'max_drawdown'),
]:
    pivot = df_bench.pivot_table(
        index='model', columns='symbol', values=col
    ).round(3)
    pivot['Avg'] = pivot.mean(axis=1).round(3)
    pivot = pivot.sort_values('Avg', ascending=(col == 'max_drawdown'))
    print(f'\n=== {metric} ===')
    display(pivot)

### Save Results

In [ ]:
df_bench.to_csv('strategy_benchmark_results.csv', index=False)
print('Saved to strategy_benchmark_results.csv')
display(df_bench[['model', 'symbol', 'avg_annual', 'sharpe_ratio', 'max_drawdown', 'status']].rename(columns={
    'avg_annual':   'ARR %',
    'sharpe_ratio': 'SHR',
    'max_drawdown': 'MDD %',
}).round(3))